In [2]:
import sys
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np

print(f"Python version: {sys.version}")
print(f"OpenCV version: {cv2.__version__}")
print(f"MediaPipe version: {mp.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

mp_pose = mp.solutions.pose
print("solutions API works!")

Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
OpenCV version: 4.13.0
MediaPipe version: 0.10.14
Pandas version: 3.0.3
Numpy version: 2.4.6
solutions API works!


In [3]:
# Start with the forehand video
video_path = "training_vid/forehand.mp4"

cap = cv2.VideoCapture(video_path)

print(f"Video opened: {cap.isOpened()}")
print(f"Total frames: {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}")
print(f"FPS: {cap.get(cv2.CAP_PROP_FPS)}")
print(f"Width: {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}")
print(f"Height: {int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")

cap.release()

Video opened: True
Total frames: 2131
FPS: 30.0
Width: 1920
Height: 1080


Analyzing one frame using MediaPipe

In [5]:
#opens video
cap = cv2.VideoCapture("training_vid/forehand.mp4")

#read a frame
success, frame = cap.read()

frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

with mp_pose.Pose() as pose:
    
    #looks at the frame looks at the 33 body landmarks
    results = pose.process(frame_rgb)
    
if results.pose_landmarks:
    print("Landmarks detected")
    print(f"Number of Landmarks: {len(results.pose_landmarks.landmark)}")
    
    #prints right wrist position
    right_wrist = results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_WRIST]
    print(f"Right wrist - x: {right_wrist.x:.3f}, y :{right_wrist.y:.3f}")
else:
    print("No landmarks detected")
    
cap.release()


Landmarks detected
Number of Landmarks: 33
Right wrist - x: 0.164, y :0.559


c:\Users\guoda\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


# Extracting each frame from the video to extract landmark locations in each frame

In [ ]:
all_landmarks = []

cap = cv2.VideoCapture("training_vid/forehand.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = 0

with mp_pose.Pose() as pose:
    while cap.isOpened():   #if vid was opened successfully
        success, frame = cap.read()     #read() returns 2 vals: bool if frame can be opened and the frame data
        
        if not success: #runs out of frames to process
            break
        
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) #converts color from BGR to RGB because mediapipe needs this format
        results = pose.process(frame_rgb)   # processes the frame for 33 landmarks
        
        if results.pose_landmarks:
            row = {"frame": frame_count, "timestamp": frame_count/fps} #adds these 2 cols into each row
            
            for landmark in mp_pose.PoseLandmark: #records the x,y,z pos and visiility of each landmark
                lm = results.pose_landmarks.landmark[landmark]
                
                #adds 4 coords of each landmark in row
                row[f"{landmark.name}_x"] = lm.x
                row[f"{landmark.name}_y"] = lm.y
                row[f"{landmark.name}_z"] = lm.z
                row[f"{landmark.name}_vis"] = lm.visibility
            
            all_landmarks.append(row)
            
        frame_count += 1
        
        if frame_count % 100 == 0:
            print(f"Processed {frame_count} frames")

cap.release()
print(f"Done! Processed {frame_count} total frames")
print(f"Frames with landmarks: {len(all_landmarks)}")
            
        
        
        
        


c:\Users\guoda\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processed 100 frames
Processed 200 frames
Processed 300 frames
Processed 400 frames
Processed 500 frames
Processed 600 frames
Processed 700 frames
Processed 800 frames
Processed 900 frames
Processed 1000 frames
Processed 1100 frames
Processed 1200 frames
Processed 1300 frames
Processed 1400 frames
Processed 1500 frames
Processed 1600 frames
Processed 1700 frames
Processed 1800 frames
Processed 1900 frames
Processed 2000 frames
Processed 2100 frames
Done! Processed 2131 total frames
Frames with landmarks: 2129


: 

In [ ]:
df = pd.DataFrame(all_landmarks)
df.to_csv = ("forehand_landmarks.csv", index = False)
